In [ ]:
import os
from tqdm import tqdm
import torch
import torchvision

from torchsummary import summary
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

from PIL import Image
import math

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32 
NUM_WORKERS = os.cpu_count()-2
USE_AMP = True # Automatic Mixed Precision

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base="data"):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]


get_folder_structure()

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root="data/trainbin/", transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
testset = torchvision.datasets.ImageFolder(root="data/testbin/", transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:
print("Follwing classes are there : \n", trainset.classes)

In [ ]:
def display_img(img, label):
    print(f"Label : {trainset.classes[label]}")
    img_valid_range = np.clip(img.numpy().transpose((1, 2, 0)), 0, 1)
    plt.imshow(img_valid_range)


# display the first image in the dataset
display_img(*trainset[0])

In [ ]:
num_classes = len(trainset.classes)
# Cargar un modelo preentrenado
model = torchvision.models.resnet152(weights=torchvision.models.ResNet152_Weights.IMAGENET1K_V2)

# Freeze all layers except the last ones
for param in model.parameters():
    param.requires_grad = False

# Modify the final layer for our classification task
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Move model to device
model = model.to(device)
# model = torch.compile(model)

In [ ]:
summary(model, (3, IMG_SIZE, IMG_SIZE), device=device.type)

In [ ]:
criterion = torch.nn.NLLLoss(reduction="sum")  #  Use NLLLoss for binary-class classification
# criterion = torch.nn.CrossEntropyLoss()  # Use CrossEntropy for multi-class classification
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-4)  # Only optimize parameters of final layer
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

In [ ]:
torch.set_num_threads(NUM_WORKERS)  # Usa todos los cores
torch.backends.cudnn.benchmark = True  # Mejora la velocidad de entrenamiento en GPU

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # loss.backward()
            # optimizer.step()
            for param in model.parameters():
                param.grad = None
            # optimizer.zero_grad()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(trainloader.dataset)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in tqdm(testloader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = correct / total
        print(f'Validation Accuracy: {val_accuracy:.4f}')

In [ ]:
train_model(model, criterion, optimizer, num_epochs=10)

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")